# BioCore — Real Face Engine on Google Colab (GPU)

Runs **InsightFace** (ArcFace 512-d embeddings + face detection) on Colab's GPU and exposes
`/embed` `/liveness` `/compare` over a **public URL** for BioCore's `RemoteFaceEngine`.
Real face recognition — not the sha512 stand-in.

**Steps**
1. **Runtime → Change runtime type → T4 GPU** (already preselected in this notebook).
2. Run **Cell 1** (installs, ~1–2 min), then **Cell 2** (loads the model + starts the server).
3. It prints `BioCore FACE_ENGINE_URL = https://xxxx.trycloudflare.com`.
4. **Send that URL to Claude.** Keep this notebook running during the demo.

*Note: this does real recognition + a basic quality gate. True anti-spoof/PAD (reject a printed
photo) is a separate model — ask Claude to add MiniFASNet.*

In [ ]:
# Cell 1 — install dependencies (~1-2 min)
!pip -q install insightface onnxruntime-gpu fastapi 'uvicorn[standard]' nest_asyncio opencv-python-headless
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print('installs done')

In [ ]:
# Cell 2 — load the real face model, start the API, open a public URL
import base64, threading, time, re, subprocess
import numpy as np, cv2
from insightface.app import FaceAnalysis
from fastapi import FastAPI, Request
import uvicorn, nest_asyncio

app_model = FaceAnalysis(name="buffalo_l",
                         providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app_model.prepare(ctx_id=0, det_size=(640, 640))
print(">> face model ready (GPU if available)")

def _decode(image):
    if image.startswith("data:"):
        image = image.split(",", 1)[1]
    arr = np.frombuffer(base64.b64decode(image), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def _largest_face(img):
    faces = app_model.get(img) if img is not None else []
    if not faces:
        return None
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))

api = FastAPI()

@api.get("/health")
def health():
    return {"status": "ok"}

@api.post("/embed")
async def embed(req: Request):
    d = await req.json()
    f = _largest_face(_decode(d["image"]))
    if f is None:
        return {"face_detected": False, "embedding": None}
    v = f.normed_embedding.astype("float32").tobytes()
    return {"face_detected": True, "embedding": base64.b64encode(v).decode(), "embedding_dim": 512}

@api.post("/liveness")
async def liveness(req: Request):
    img = _decode((await req.json())["image"])
    f = _largest_face(img)
    if f is None:
        return {"live": False, "detail": "no_face"}
    area = (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1])/(img.shape[0]*img.shape[1])
    sharp = cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
    ok = area > 0.02      # webcam-friendly; NOT anti-spoof
    return {"live": bool(ok), "detail": f"area={area:.3f} sharp={sharp:.0f}"}

@api.post("/compare")
async def compare(req: Request):
    d = await req.json()
    a = np.frombuffer(base64.b64decode(d["a"]), np.float32)
    b = np.frombuffer(base64.b64decode(d["b"]), np.float32)
    if a.size == 0 or b.size == 0:
        return {"score": 0.0, "match": False}
    cos = float(np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9))
    return {"score": cos, "match": cos >= 0.40}

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(api, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True).start()
time.sleep(3)

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n\n=====================================================")
print("  BioCore FACE_ENGINE_URL =", url)
print("  -> send this URL to Claude to wire it in.")
print("  (keep this notebook running for the whole demo)")
print("=====================================================\n")